In [263]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score ,classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("train.txt",sep=';',header=None,names=['Text','Emotion'])

In [3]:
df.head()

,Text,Emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [4]:
df.shape

(16000, 2)

In [5]:
df.isnull().sum()

Text       0
Emotion    0
dtype: int64

In [6]:
unique_emotions = df['Emotion'].unique()
emotion_number = {}
i = 0
for emo in unique_emotions:
    emotion_number[emo] = i
    i += 1
df['Emotion'] = df["Emotion"].map(emotion_number)

In [7]:
df.head()

,Text,Emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [8]:
df['Text'] = df["Text"].apply(lambda x : x.lower())

In [9]:
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))
df['Text']  = df["Text"].apply(remove_punc)

In [10]:
def remove_numbers(txt):
    new = ''
    for i in txt:
        if not i.isdigit():
            new = new + i
    return new

df['Text'] = df['Text'].apply(remove_numbers)

In [11]:
def remove_emojis(txt):
    new = ""
    for i in txt:
        if i.isascii():
            new = new + i
    return new
    
df['Text'] = df['Text'].apply(remove_emojis)

In [12]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\saula\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\saula\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
stop_words = set(stopwords.words('english'))

In [14]:
len(stop_words)

198

In [15]:
def remove(txt):
    words = word_tokenize(txt)
    cleaned = []
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return " ".join(cleaned)

df['Text'] = df['Text'].apply(remove)

In [16]:
df.head()

,Text,Emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [156]:
X_train , X_test , y_train , y_test = train_test_split(df['Text'],df['Emotion'],test_size=0.2,random_state=42)

In [238]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [240]:
#NAIVE BIAS
nb_model = MultinomialNB()
nb_model.fit(X_train_bow,y_train)
y_pred = nb_model.predict(X_test_bow)
accuracy = accuracy_score(y_pred,y_test)
print(f"Accuracy : {accuracy}")

Accuracy : 0.7678125


In [242]:
#Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_bow,y_train)
y_pred = lr_model.predict(X_test_bow)
accuracy = accuracy_score(y_pred,y_test)
print(f"Accuracy : {accuracy}")

Accuracy : 0.88875


In [244]:
#Support Vector Machine
svm_model = SVC()
svm_model.fit(X_train_bow,y_train)
y_pred = svm_model.predict(X_test_bow)
accuracy = accuracy_score(y_pred,y_test)
print(f"Accuracy : {accuracy}")

Accuracy : 0.8225


In [248]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [250]:
#NAIVE BIAS
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf,y_train)
y_pred = nb_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_pred,y_test)
print(f"Accuracy : {accuracy}")

Accuracy : 0.6609375


In [254]:
#Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf,y_train)
y_pred = lr_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_pred,y_test)
print(f"Accuracy : {accuracy}")

Accuracy : 0.8615625


In [252]:
#Support Vector Machine
svm_model = SVC()
svm_model.fit(X_train_tfidf,y_train)
y_pred = svm_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_pred,y_test)
print(f"Accuracy : {accuracy}")

Accuracy : 0.8515625


In [184]:
classifier = GridSearchCV(lr_model,{
    'solver': ['liblinear', 'saga'],
    'penalty': ['l1', 'l2'],
    'C': [0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced']
}, cv=3,return_train_score=False)

classifier.fit(X_train_bow,y_train)
results = pd.DataFrame(classifier.cv_results_)
results[["param_solver","param_penalty","param_C","param_class_weight","mean_test_score"]]

results['mean_test_score']

,param_solver,param_penalty,param_C,param_class_weight,mean_test_score
0,liblinear,l1,0.01,None,0.339141
1,saga,l1,0.01,None,0.339141
2,liblinear,l2,0.01,None,0.527891
3,saga,l2,0.01,None,0.557813
4,liblinear,l1,0.01,balanced,0.352578
5,saga,l1,0.01,balanced,0.137969
6,liblinear,l2,0.01,balanced,0.703046
7,saga,l2,0.01,balanced,0.719844
8,liblinear,l1,0.10,None,0.841015
9,saga,l1,0.10,None,0.831250


# Hyper Parameter Tuning

In [257]:
X_train , X_test , y_train , y_test = train_test_split(df['Text'],df['Emotion'],test_size=0.2,random_state=42)

bow_vectorizer = CountVectorizer(ngram_range=(1,2))
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

#Logistic Regression
lr_model = LogisticRegression(penalty='l1',C=1.00,random_state=42,solver='liblinear',max_iter=1000)
lr_model.fit(X_train_bow,y_train)
y_pred = lr_model.predict(X_test_bow)
accuracy = accuracy_score(y_pred,y_test)
print(f"Accuracy : {round(accuracy,2)}")

Accuracy : 0.91


In [259]:
print(classification_report(y_pred,y_test))

              precision    recall  f1-score   support

           0       0.94      0.95      0.95       931
           1       0.90      0.89      0.89       432
           2       0.83      0.85      0.84       291
           3       0.74      0.87      0.80        97
           4       0.88      0.87      0.88       403
           5       0.94      0.92      0.93      1046

    accuracy                           0.91      3200
   macro avg       0.87      0.89      0.88      3200
weighted avg       0.91      0.91      0.91      3200



In [267]:
import joblib
joblib.dump(lr_model, "emotion_model.pkl")
joblib.dump(bow_vectorizer, "bow_vectorizer.pkl")

['bow_vectorizer.pkl']